In [ ]:
import itertools
from bg_atlasapi.bg_atlas import BrainGlobeAtlas

atlas = BrainGlobeAtlas("allen_mouse_50um")
annot = atlas.annotation

vox = np.asarray(probe["data"]["sites_vox"]).reshape(-1, 3)
labels = np.asarray(probe["data"]["sites_label"]).reshape(-1)

print("BrainGlobe annotation shape:", annot.shape)
print("HERBS voxel min:", vox.min(axis=0))
print("HERBS voxel max:", vox.max(axis=0))

perms = list(itertools.permutations([0, 1, 2]))

results = []

for perm in perms:
    v = vox[:, perm].copy()

    for flips in itertools.product([False, True], repeat=3):
        vv = v.copy()

        for ax, do_flip in enumerate(flips):
            if do_flip:
                vv[:, ax] = annot.shape[ax] - 1 - vv[:, ax]

        valid = np.all((vv >= 0) & (vv < np.array(annot.shape)), axis=1)

        if valid.sum() == 0:
            continue

        vv_valid = vv[valid].astype(int)
        labels_valid = labels[valid]

        atlas_labels = annot[
            vv_valid[:, 0],
            vv_valid[:, 1],
            vv_valid[:, 2]
        ]

        match = atlas_labels == labels_valid
        score = match.mean()

        results.append({
            "perm": perm,
            "flips": flips,
            "valid_n": valid.sum(),
            "score": score,
            "matches": match.sum()
        })

results = sorted(results, key=lambda x: x["score"], reverse=True)

for r in results[:10]:
    print(r)

NameError: name 'probe' is not defined

In [ ]:
import numpy as np
import vedo
vedo.settings.default_backend = "vtk"

from bg_atlasapi.bg_atlas import BrainGlobeAtlas
from brainrender import Scene
from brainrender.actors import Points

atlas = BrainGlobeAtlas("allen_mouse_50um")
shape = np.array(atlas.annotation.shape)  # (264, 160, 228)

vox = np.asarray(probe["data"]["sites_vox"]).reshape(-1, 3)

# HERBS -> BrainGlobe voxel coordinates
v = vox[:, (1, 2, 0)].copy()
v[:, 0] = shape[0] - 1 - v[:, 0]
v[:, 1] = shape[1] - 1 - v[:, 1]

# flip hemisphere / left-right axis
v[:, 2] = shape[2] - 1 - v[:, 2]

pts = v * 50

scene = Scene(atlas_name="allen_mouse_50um", inset=False)

scene.add_brain_region("PAG", alpha=0.25)
scene.add_brain_region("SCm", alpha=0.25)

scene.add(Points(pts, radius=25, name="HERBS probe sites"))

scene.render()